# A股量化交易平台 · Jupyter 快速上手

**前提**：这个 notebook 必须用 **Python 3.11 及以上** 的内核运行（akshare 最新版的要求）。
Anaconda 用户的配置方法见项目 README 的「Jupyter Notebook」一节；配好后在右上角（或菜单 Kernel → Change kernel）选择 **Python 3.12 (quant)**。

从上到下依次运行每个格子（Shift + Enter）。

In [ ]:
# 1. 检查 Python 版本，并把项目目录加入导入路径
import sys, os
from pathlib import Path

print("Python", sys.version.split()[0])
assert sys.version_info >= (3, 11), "内核版本太低：请切换到 Python 3.11+ 的内核（见 README 的 Jupyter 一节）"

ROOT = Path.cwd() if (Path.cwd() / "quant").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("项目目录:", ROOT)

In [ ]:
# 2. 安装依赖（只需第一次运行；先用清华镜像，失败再用官方源）
!"{sys.executable}" -m pip install -q -r "{ROOT / 'requirements.txt'}" matplotlib -i https://pypi.tuna.tsinghua.edu.cn/simple || "{sys.executable}" -m pip install -q -r "{ROOT / 'requirements.txt'}" matplotlib

## 启动实时看板

运行下面的格子后会自动打开浏览器 http://127.0.0.1:8000 。平台在后台线程运行，notebook 可以继续使用。

- `start_platform()`：仿真行情（任何时间都能用）
- `start_platform(feed="akshare")`：东方财富准实时行情（需中国大陆网络、交易时段）

In [ ]:
from quant.notebook import start_platform, stop_platform, backtest, plot_backtest, download, load_bars

engine = start_platform()

In [ ]:
# 3. 在 notebook 里直接查看引擎状态 / 最新行情 / 账户
import time
import pandas as pd

for _ in range(60):          # 刚启动时可能还没开盘，最多等 30 秒拿到第一批行情
    if engine.ticks:
        break
    time.sleep(0.5)
print(engine.status()["feed"]["state"], "行情时间", engine.feed.now())
pd.DataFrame([t.to_dict() for t in engine.ticks.values()],
             columns=["symbol", "name", "last", "pre_close", "volume", "limit_up", "limit_down"])

In [ ]:
engine.broker.account().to_dict()

## 回测

本地没有真实日线时，`synthetic=True` 会用合成数据先跑通流程（**合成数据不代表真实行情，结果没有参考意义**）。
在中国大陆网络下，先运行下一节的 `download` 下载真实日线，再把 `synthetic` 去掉。

In [ ]:
res = backtest("dual_ma", ["600000", "000001", "300750"], start="2022-01-01", end="2025-06-30",
               params={"fast": 5, "slow": 20, "position_pct": 0.2}, synthetic=True)
plot_backtest(res, "双均线");

In [ ]:
# 成交明细
pd.DataFrame([t.to_dict() for t in res.trades]).tail(10)

## 下载真实数据（需中国大陆网络）

In [ ]:
try:
    print(download(["600000", "000001", "300750"], start="20200101"))
    display(load_bars("600000").tail())
except Exception as e:
    print("下载失败（海外网络无法访问东方财富，或被限流）：", e)

## 在 notebook 里写自己的策略

继承 `Strategy`，重写 `on_bar`，直接把类传给 `backtest`。同一份代码也能放进 `config.yaml` 跑实时模拟盘。

In [ ]:
from quant.strategy import Strategy, BarSeries

class RsiStrategy(Strategy):
    default_params = {"n": 14, "low": 30, "high": 70, "position_pct": 0.25}

    def on_init(self):
        self.series = {s: BarSeries(self.n + 1) for s in self.symbols}

    def on_bar(self, bar):
        s = self.series[bar.symbol]
        s.update(bar)
        if not s.ready:
            return
        rsi = s.rsi(self.n)
        if rsi < self.low and self.pos(bar.symbol) == 0:
            self.buy_value(bar.symbol, self.equity * self.position_pct)
        elif rsi > self.high:
            self.sell_all(bar.symbol)

res2 = backtest(RsiStrategy, ["600000", "000001"], start="2022-01-01", synthetic=True)
plot_backtest(res2, "RSI 反转");

## 停止平台

In [ ]:
stop_platform()